# 01 — Data preprocessing (MoMentS)

Builds the tables used by later notebooks. Everything is written to `data-private/processed/` (git-ignored).

Inputs (all under `data-private/raw/`):
- `moments_github/` — public MoMentS repo, commit in `SOURCE.txt` (questions, options, timestamps, categories, cue tags; validation keys).
- `moments_questions_v4_test_keys.json` — **private** author-provided test answer keys.
- `all_trimmed_trxs/<question_id>/{ti_tj,t0_tj}_transcript.json` — author WhisperX transcripts (times are film-absolute seconds).
- `all_videos/<movie_title>.mp4` — author-provided films.

Gold answers are kept in a **separate** table (`answer_keys.parquet`) and never joined into the question table.

In [11]:
import json
from pathlib import Path
import pandas as pd
import av

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data-private" / "raw"
GH = RAW / "moments_github"
TRX_DIR = RAW / "all_trimmed_trxs"
VIDEO_DIR = RAW / "all_videos"
OUT = ROOT / "data-private" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

CATEGORIES = ["Emotions", "Intentions", "Beliefs", "Desires", "Knowledge", "Percepts", "Non-literal communication"]
CUES = {"Face Expression and Gaze": "cue_face", "Body Language": "cue_body", "Speech-related": "cue_speech"}

print("ROOT:", ROOT)
print((GH / "SOURCE.txt").read_text())

ROOT: /home/sactive/Projects/ai-alignment-demands-emotions
repo: https://github.com/villacu/MoMentS
commit: c6a94c6f9390e0474e2bfde0320278a8fd62c1d0
downloaded: 2026-09-26T11:23:22+03:00



## 1. Questions (public repo, no answers)

In [12]:
test_q = json.loads((GH / "data/test/moments_test_questions.json").read_text())
val_q = json.loads((GH / "data/validation/moments_validation_questions.json").read_text())
for q in test_q: q["split"] = "test"
for q in val_q: q["split"] = "validation"
raw_q = test_q + val_q

assert not any("answer" in k.lower() or "correct" in k.lower() for q in raw_q for k in q), "answer field found in question file"
print("test:", len(test_q), "| validation:", len(val_q), "| total:", len(raw_q))
print("fields:", sorted(raw_q[0].keys()))

test: 2010 | validation: 325 | total: 2335
fields: ['assigned_categories', 'movie_title', 'multimodal_cues', 'options', 'question', 'question_id', 'split', 't_0', 't_i', 't_j', 'video_length', 'video_url']


In [13]:
def cue_state(v):
    if v is None: return "null"
    return "empty_list" if len(v) == 0 else "list"

rows = []
for q in raw_q:
    cues = q["multimodal_cues"] or []
    unknown = set(cues) - set(CUES)
    assert not unknown, unknown
    row = {
        "question_id": q["question_id"], "split": q["split"], "movie_title": q["movie_title"],
        "question": q["question"],
        **{f"option_{k}": q["options"][k] for k in "ABCD"},
        "t_0": q["t_0"], "t_i": q["t_i"], "t_j": q["t_j"],
        "focused_duration_s": q["t_j"] - q["t_i"],
        "video_length_dataset_s": q["video_length"], "video_url": q["video_url"],
        "categories": list(q["assigned_categories"]),
        "cues": list(cues), "cue_field_state": cue_state(q["multimodal_cues"]),
    }
    for c in CATEGORIES:
        row["cat_" + c.split()[0].lower().replace("-", "_")] = c in q["assigned_categories"]
    for name, col in CUES.items():
        row[col] = name in cues
    row["cue_visual_any"] = row["cue_face"] or row["cue_body"]
    rows.append(row)

questions = pd.DataFrame(rows)
assert questions["question_id"].is_unique
assert questions[["option_A", "option_B", "option_C", "option_D"]].notna().all().all()
assert set(c for cs in questions["categories"] for c in cs) <= set(CATEGORIES)
questions.head(3).drop(columns=["question", "option_A", "option_B", "option_C", "option_D"])

,question_id,split,movie_title,t_0,t_i,t_j,focused_duration_s,video_length_dataset_s,video_url,categories,...,cat_intentions,cat_beliefs,cat_desires,cat_knowledge,cat_percepts,cat_non_literal,cue_face,cue_body,cue_speech,cue_visual_any
0,zT2dD,test,A NEW MAN,0.0,102.659524,144.268208,41.608684,537.66,https://www.youtube.com/watch?v=k2jX6XqcIp0,[Emotions],...,False,False,False,False,False,False,False,False,False,False
1,_Um8G,test,A NEW MAN,0.0,66.525731,88.388824,21.863093,537.66,https://www.youtube.com/watch?v=k2jX6XqcIp0,[Desires],...,False,False,True,False,False,False,True,False,False,True
2,7Jz0B,test,A NEW MAN,0.0,295.376795,317.346969,21.970175,537.66,https://www.youtube.com/watch?v=k2jX6XqcIp0,[Beliefs],...,False,True,False,False,False,False,False,False,False,False


## 2. Answer keys (separate table — never joined into model inputs)

In [14]:
test_keys = pd.DataFrame(json.loads((RAW / "moments_questions_v4_test_keys.json").read_text())).assign(split="test", key_source="author_private")
val_keys = pd.DataFrame(json.loads((GH / "data/validation/moments_validation_keys.json").read_text())).assign(split="validation", key_source="public_repo")
answer_keys = pd.concat([test_keys, val_keys], ignore_index=True)

assert answer_keys["question_id"].is_unique
assert set(answer_keys["question_id"]) == set(questions["question_id"]), "keys and questions do not match"
assert set(answer_keys["correct_answer_key"]) == set("ABCD")
print(answer_keys.groupby("split")["correct_answer_key"].value_counts().unstack())

correct_answer_key    A    B    C    D
split                                 
test                517  539  492  462
validation           76   85   78   86


## 3. Transcripts (author WhisperX output, one row per segment)

Segment times are film-absolute seconds. Some segments at the window edges extend slightly past `[t_i, t_j]` or start exactly at `t_i` (author trimming); they are kept as-is.

In [15]:
WINDOWS = {"focused": "ti_tj_transcript.json", "full": "t0_tj_transcript.json"}
seg_rows = []
for qid in questions["question_id"]:
    for window, fname in WINDOWS.items():
        for pos, s in enumerate(json.loads((TRX_DIR / qid / fname).read_text())):
            seg_rows.append({
                "question_id": qid, "window": window, "position": pos, "segment_index": s["index"],
                "start": s["start"], "end": s["end"], "text": s["text"],
                "n_words": len(s["words"]), "avg_logprob": s["avg_logprob"],
                "word_texts": [w["word"] for w in s["words"]], "word_scores": [w.get("score") for w in s["words"]],
                "word_starts": [w["start"] for w in s["words"]], "word_ends": [w["end"] for w in s["words"]],
            })
transcripts = pd.DataFrame(seg_rows)

extra_dirs = {p.name for p in TRX_DIR.iterdir()} - set(questions["question_id"])
print("transcript folders without a question:", len(extra_dirs))
print(transcripts.groupby("window").agg(segments=("text", "size"), questions=("question_id", "nunique")))
focused_n = transcripts[transcripts.window == "focused"].groupby("question_id").size()
questions["focused_n_segments"] = questions["question_id"].map(focused_n).fillna(0).astype(int)
questions["focused_n_words"] = questions["question_id"].map(transcripts[transcripts.window == "focused"].groupby("question_id")["n_words"].sum()).fillna(0).astype(int)
print("questions with EMPTY focused transcript:", (questions.focused_n_segments == 0).sum())

transcript folders without a question: 0
         segments  questions
window                      
focused     24205       2097
full       188689       2310
questions with EMPTY focused transcript: 238


## 4. Transcript rendering — the single function used for T and VT

Default format (our choice; the paper does not specify one): one line per WhisperX segment, surrounding whitespace stripped, no timestamps, no speaker labels (none exist in the files). The same string is used in both conditions.

In [16]:
_focused_texts = (transcripts[transcripts.window == "focused"].sort_values(["question_id", "position"])
                  .groupby("question_id")["text"].apply(list).to_dict())

def render_transcript(question_id):
    """Exact focused-window transcript text given to the model. Deterministic; identical for T and VT."""
    return "\n".join(t.strip() for t in _focused_texts.get(question_id, []) if t.strip())

questions["transcript_focused"] = questions["question_id"].map(render_transcript)
example = questions[questions.focused_n_segments > 0].iloc[0]
print(example.question_id, "\n---\n" + example.transcript_focused)

zT2dD 
---
I'm really sorry I'm late.
I know we said nine, but I had a nightmare morning with the kid.
What's 10 minutes?
Forget it.
Well, is your kid okay?
Yeah.
You know, we should just start a kindergarten.
You know the deal.
I mean, you got enough on your own plate without worrying about my problems.
Let's get down to business.


## 5. Videos (one row per local file)

All 175 local `.mp4` files are listed; `in_dataset` marks the 168 films used by questions. The dataset `video_length` is compared with the local file (a length mismatch alone does not mean misalignment; see section 6).

In [17]:
vid_rows = []
missing = sorted(set(questions["movie_title"]) - {p.stem for p in VIDEO_DIR.glob("*.mp4")})
assert not missing, f"missing videos: {missing}"
for path in sorted(VIDEO_DIR.glob("*.mp4")):
    title = path.stem
    with av.open(str(path)) as c:
        vs = c.streams.video[0]
        vid_rows.append({
            "movie_title": title, "video_path": str(path.relative_to(ROOT)),
            "local_duration_s": c.duration / 1e6,
            "width": vs.codec_context.width, "height": vs.codec_context.height,
            "fps": float(vs.average_rate), "codec": vs.codec_context.name,
            "has_audio": len(c.streams.audio) > 0, "encoder": c.metadata.get("encoder", ""),
        })
videos = pd.DataFrame(vid_rows)
film_meta = questions.groupby("movie_title").agg(video_length_dataset_s=("video_length_dataset_s", "first"),
                                                 max_t_j=("t_j", "max"), n_questions=("question_id", "size"),
                                                 split=("split", "first"))
videos = videos.merge(film_meta, on="movie_title", how="left")
videos["in_dataset"] = videos.movie_title.isin(questions.movie_title)
videos["duration_diff_s"] = videos.local_duration_s - videos.video_length_dataset_s
videos["t_j_beyond_video"] = videos.max_t_j > videos.local_duration_s

print("local files:", len(videos), "| in dataset:", videos.in_dataset.sum(), "| not used by any question:", videos.loc[~videos.in_dataset, "movie_title"].tolist())
print("|duration diff| > 1 s:", (videos.duration_diff_s.abs() > 1).sum())
print("films where some t_j is past the end of the local video:", videos.loc[videos.t_j_beyond_video, "movie_title"].tolist())
videos.sort_values("duration_diff_s", key=abs, ascending=False).head(10)

local files: 175 | in dataset: 168 | not used by any question: ['PRETTY PEOPLE', 'STOP 2', 'STRANGERS', 'THE CRITIC', 'THE GESTURE AND THE WORD', 'TICTOC', 'WINNING IN AMERICA']
|duration diff| > 1 s: 45
films where some t_j is past the end of the local video: ['STOP']


,movie_title,video_path,local_duration_s,width,height,fps,codec,has_audio,encoder,video_length_dataset_s,max_t_j,n_questions,split,in_dataset,duration_diff_s,t_j_beyond_video
167,VIRAL,data/raw/all_videos/VIRAL.mp4,651.459048,640,360,23.976024,h264,True,,556.181,321.712442,4.0,test,True,95.278048,False
119,REBEL,data/raw/all_videos/REBEL.mp4,882.636916,640,360,24.000000,h264,True,,901.931,780.800464,11.0,test,True,-19.294084,False
88,MAN TO MAN,data/raw/all_videos/MAN TO MAN.mp4,730.105034,640,360,23.976024,h264,True,,720.872,617.003124,9.0,test,True,9.233034,False
86,MAGICIAN,data/raw/all_videos/MAGICIAN.mp4,468.067846,854,480,25.000000,h264,True,Lavf59.27.100,459.326,378.283348,18.0,test,True,8.741846,False
24,BURNT TOAST AND CIGARETTES,data/raw/all_videos/BURNT TOAST AND CIGARETTES...,1148.691156,640,360,25.000000,h264,True,,1140.011,1095.338311,19.0,test,True,8.680156,False
109,ONE DAY NOTICE,data/raw/all_videos/ONE DAY NOTICE.mp4,1346.850249,640,360,23.976024,h264,True,,1338.219,1252.257220,10.0,test,True,8.631249,False
137,THE CHOKE,data/raw/all_videos/THE CHOKE.mp4,1380.588844,640,360,24.000000,h264,True,,1372.223,1327.905406,9.0,test,True,8.365844,False
92,MAXIMUS,data/raw/all_videos/MAXIMUS.mp4,964.858776,854,480,24.000000,h264,True,Lavf59.27.100,956.587,806.648348,13.0,test,True,8.271776,False
144,THE GUARD,data/raw/all_videos/THE GUARD.mp4,925.849252,640,360,23.976024,h264,True,Google,934.059,889.549627,13.0,test,True,-8.209748,False
166,UNDER DARKNESS,data/raw/all_videos/UNDER DARKNESS.mp4,1227.035283,640,360,23.976024,h264,True,,1218.898,1103.754376,14.0,test,True,8.137283,False


## 6. Video alignment check (transcript timings ↔ local video)

Our videos were downloaded separately from the transcripts, so we check that the transcript times match the local files.
For each film, we take its longest full-window transcript (hundreds of words) and compare "someone is speaking"
(from word times) with the loudness of the film's own audio, for shifts of −15…+15 s (`src/video_alignment.py`).

- `align_shift_s`: the best shift. 0 means the transcript matches the video.
- `align_corr`: the correlation at that shift. Values near 0 mean the check cannot tell (little speech, loud music).

The audio is used only for this check, never as model input. Nothing is excluded here.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import sys
sys.path.insert(0, str(ROOT / "src"))
from video_alignment import film_shift

full = transcripts[transcripts.window == "full"]
jobs, titles = [], []
for title, g in questions.groupby("movie_title"):
    qid = g.loc[g.t_j.idxmax(), "question_id"]  # this question's full window [t_0, t_j] covers the most of the film
    seg = full[full.question_id == qid]
    word_times = [(s, e) for starts, ends in zip(seg.word_starts, seg.word_ends) for s, e in zip(starts, ends)]
    jobs.append((VIDEO_DIR / f"{title}.mp4", word_times, g.t_j.max()))
    titles.append((title, len(word_times)))

with ProcessPoolExecutor(8) as ex:
    shifts = list(ex.map(film_shift, jobs))
align = pd.DataFrame([(t, n, s, r) for (t, n), (s, r) in zip(titles, shifts)],
                     columns=["movie_title", "align_n_words", "align_shift_s", "align_corr"])
videos = videos.drop(columns=align.columns[1:], errors="ignore").merge(align, on="movie_title", how="left")

print("films with best shift 0 s:", (align.align_shift_s.abs() <= 0.5).sum(), "of", len(align))
print(align[align.align_shift_s.abs() > 0.5].round(2).to_string(index=False))

In [ ]:
print("films where some question's t_j is past the end of the local video:")
print(videos.loc[videos.t_j_beyond_video, ["movie_title", "local_duration_s", "max_t_j"]].round(1).to_string(index=False))

## 7. Summary checks against the paper

In [20]:
checks = pd.DataFrame([
    ("questions (test + validation)", len(questions), 2335),
    ("films", questions.movie_title.nunique(), 168),
    ("test questions", (questions.split == "test").sum(), 2010),
    ("validation questions / films", f"{(questions.split=='validation').sum()} / {questions[questions.split=='validation'].movie_title.nunique()}", "325 / 13"),
    ("mean focused length (s)", round(questions.focused_duration_s.mean(), 2), "42.44 ± 55.5"),
    ("SD focused length (s)", round(questions.focused_duration_s.std(), 2), "55.5"),
    ("mean full length (s)", round((questions.t_j - questions.t_0).mean(), 2), "388.47 ± 262.3"),
    ("mean question length (words)", round(questions.question.str.split().str.len().mean(), 2), "12.64 ± 4.2"),
], columns=["check", "ours", "paper"])
display(checks)
print(questions[[c for c in questions if c.startswith("cat_")]].sum().rename("n_questions").to_string())
print(questions[["cue_face", "cue_body", "cue_speech", "cue_visual_any"]].sum().to_string())
print(questions.cue_field_state.value_counts().to_string())

,check,ours,paper
0,questions (test + validation),2335,2335
1,films,168,168
2,test questions,2010,2010
3,validation questions / films,325 / 13,325 / 13
4,mean focused length (s),42.37,42.44 ± 55.5
5,SD focused length (s),55.55,55.5
6,mean full length (s),387.94,388.47 ± 262.3
7,mean question length (words),12.64,12.64 ± 4.2


cat_emotions        598
cat_intentions     1024
cat_beliefs         382
cat_desires         389
cat_knowledge       334
cat_percepts        319
cat_non_literal     215
cue_face          1438
cue_body          1177
cue_speech         985
cue_visual_any    1846
cue_field_state
list          2099
null           205
empty_list      31


## 8. Save

In [21]:
questions.to_parquet(OUT / "questions.parquet", index=False)
answer_keys.to_parquet(OUT / "answer_keys.parquet", index=False)
transcripts.to_parquet(OUT / "transcripts.parquet", index=False)
videos.to_parquet(OUT / "videos.parquet", index=False)
assert "correct_answer_key" not in questions.columns
for f in sorted(OUT.glob("*.parquet")):
    print(f.relative_to(ROOT), f"{f.stat().st_size/1e6:.1f} MB")

data/processed/answer_keys.parquet 0.0 MB
data/processed/questions.parquet 1.0 MB
data/processed/transcripts.parquet 9.7 MB
data/processed/videos.parquet 0.0 MB
